# 4. Results collection and plots generation

In [ ]:
import os
import re
import sys
import argparse
import subprocess
import matplotlib
import time
import random
import string
import shlex
import shutil
import glob
import pickle
import csv
import operator
import joblib
import pandas as pd
import numpy as np
from itertools import groupby
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_curve
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from collections import defaultdict
from sklearn.ensemble import BaggingClassifier
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
from Bio import SeqIO
from sklearn.feature_selection import RFE
from sklearn import preprocessing
import pybedtools as pbt
import pyBigWig as pbw
from datetime import date
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from itertools import chain
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from sklearn.ensemble import AdaBoostClassifier
from adjustText import adjust_text
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from pylab import *
%matplotlib inline
from archipielago.config import load_config
cfg = load_config()


In [ ]:

root = os.getcwd()
basicdir = os.path.abspath('Release/TF-ML/')
outputdir = os.path.abspath('Release/TF-ML/outputdir')
train_dir = os.path.abspath('Release/TF-ML/train/') 
test_dir = os.path.abspath('Release/TF-ML/test/') 

if not os.path.exists(basicdir):
    os.makedirs(basicdir)
if not os.path.exists(outputdir):
    os.makedirs(outputdir)
if not os.path.exists(train_dir):
    os.makedirs(train_dir)
if not os.path.exists(test_dir):
    os.makedirs(test_dir)


pwmdir_mono = os.path.abspath('./hocomoco11/models/pwm/mono/all')
pwmdir_di = os.path.abspath('./hocomoco11/models/pwm/di/all')


files_pwm_HUMAN_mono = []
files_pwm_MOUSE_mono = []

for d in os.listdir(pwmdir_mono):
    d = pwmdir_mono + "/" + d 
    if os.path.isdir(d):
        os.chdir(d)
        if d.split("_")[-1] == "HUMAN":
            files_pwm_HUMAN_mono.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.pwm']) 
        if d.split("_")[-1] == "MOUSE":
            files_pwm_MOUSE_mono.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.pwm'])
    os.chdir(pwmdir_mono)

print(len(files_pwm_HUMAN_mono))
print(len(files_pwm_MOUSE_mono))


files_pwm_HUMAN_di = []
files_pwm_MOUSE_di = []

for d in os.listdir(pwmdir_di):
    d = pwmdir_di + "/" + d
    if os.path.isdir(d): 
        os.chdir(d) 
        if d.split("_")[-1] == "HUMAN":
            files_pwm_HUMAN_di.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.dpwm']) 
        if d.split("_")[-1] == "MOUSE":
            files_pwm_MOUSE_di.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.dpwm'])
    os.chdir(pwmdir_di) 

print(len(files_pwm_HUMAN_di))
print(len(files_pwm_MOUSE_di))


def remove_duplicates(l):
    return list(set(l))

print("We have {N} TF before filtration of mono".format(N=len(remove_duplicates([x[0].split("~")[0].split("_")[0] for x in files_pwm_MOUSE_mono + files_pwm_HUMAN_mono]))))
print("We have {N} TF before filtration of di".format(N=len(remove_duplicates([x[0].split("~")[0].split("_")[0] for x in files_pwm_MOUSE_di + files_pwm_HUMAN_di]))))

In [ ]:
TF_list = []
for i in files_pwm_MOUSE_mono + files_pwm_HUMAN_mono:
    experiment_id = [x.split("~")[2].split(".")[1] for x in i] 
    d = {}
    for j in experiment_id:
        if j in d.keys(): 
            d[j] += 1
        else:
            d[j] = 1
    if len(d.keys()) >= 2: 
        TF_list.append(i)

dict_of_names = {}
for i in TF_list:
    i = i[0].split("~")[0].split("_")[0]
    #print(i)
    if i in dict_of_names.keys():
        dict_of_names[i] += 1
    else:
        dict_of_names[i] = 1

good_names_list_mono = []  
for i in dict_of_names.keys():
    if dict_of_names[i] >= 2:
        good_names_list_mono.append(i)

print(good_names_list_mono[:10])


files_pwm_MOUSE_filtered = []
files_pwm_HUMAN_filtered = []

for i in files_pwm_MOUSE_mono:
    if i[0].split("~")[0].split("_")[0] in good_names_list_mono:
        files_pwm_MOUSE_filtered.append(i)

for i in files_pwm_HUMAN_mono:
    if i[0].split("~")[0].split("_")[0] in good_names_list_mono:
        files_pwm_HUMAN_filtered.append(i)


TF_list = []
for i in files_pwm_MOUSE_di + files_pwm_HUMAN_di:
    experiment_id = [x.split("~")[2].split(".")[1] for x in i] 
    d = {}
    for j in experiment_id:
        if j in d.keys():
            d[j] += 1
        else:
            d[j] = 1
    if len(d.keys()) >= 2:
        TF_list.append(i)

dict_of_names = {}
for i in TF_list:
    i = i[0].split("~")[0].split("_")[0]
    if i in dict_of_names.keys():
        dict_of_names[i] += 1
    else:
        dict_of_names[i] = 1

good_names_list_di = []
for i in dict_of_names.keys():
    if dict_of_names[i] >= 2:
        good_names_list_di.append(i)

print(good_names_list_di[:10])

files_pwm_MOUSE_filtered = []
files_pwm_HUMAN_filtered = []

for i in files_pwm_MOUSE_di:
    if i[0].split("~")[0].split("_")[0] in good_names_list_di:
        files_pwm_MOUSE_filtered.append(i)

for i in files_pwm_HUMAN_di:
    if i[0].split("~")[0].split("_")[0] in good_names_list_di:
        files_pwm_HUMAN_filtered.append(i)
        
good_names_list = set(good_names_list_mono) & set(good_names_list_di)

selected_short_names_list = good_names_list
selected_HUMAN = [x + "_HUMAN" for x in good_names_list]
selected_MOUSE = [x + "_MOUSE" for x in good_names_list]

print("We got {N} TF".format(N=len(selected_short_names_list)))
selected_full_names_list = selected_HUMAN + selected_MOUSE

In [ ]:
selected_HUMAN = ['ANDR_HUMAN',
 'AP2A_HUMAN',
 'BATF_HUMAN',
 'CEBPB_HUMAN',
 'COE1_HUMAN',
 'CTCF_HUMAN',
 'E2F4_HUMAN',
 'ERG_HUMAN',
 'ESR1_HUMAN',
 'FLI1_HUMAN',
 'GATA1_HUMAN',
 'GATA2_HUMAN',
 'GATA3_HUMAN',
 'GCR_HUMAN',
 'HNF4A_HUMAN',
 'IRF1_HUMAN',
 'IRF4_HUMAN',
 'JUND_HUMAN',
 'MAFK_HUMAN',
 'MAX_HUMAN',
 'MYC_HUMAN',
 'MYOD1_HUMAN',
 'P53_HUMAN',
 'PPARG_HUMAN',
 'PRGR_HUMAN',
 'REST_HUMAN',
 'RUNX1_HUMAN',
 'RXRA_HUMAN',
 'SOX2_HUMAN',
 'SPI1_HUMAN',
 'SRF_HUMAN',
 'STA5A_HUMAN',
 'STAT1_HUMAN',
 'STAT3_HUMAN',
 'TAL1_HUMAN',
 'TF65_HUMAN',
 'TFE2_HUMAN',
 'USF2_HUMAN']

selected_MOUSE = ['ANDR_MOUSE',
 'AP2A_MOUSE',
 'BATF_MOUSE',
 'CEBPB_MOUSE',
 'COE1_MOUSE',
 'CTCF_MOUSE',
 'E2F4_MOUSE',
 'ERG_MOUSE',
 'ESR1_MOUSE',
 'FLI1_MOUSE',
 'GATA1_MOUSE',
 'GATA2_MOUSE',
 'GATA3_MOUSE',
 'GCR_MOUSE',
 'HNF4A_MOUSE',
 'IRF1_MOUSE',
 'IRF4_MOUSE',
 'JUND_MOUSE',
 'MAFK_MOUSE',
 'MAX_MOUSE',
 'MYC_MOUSE',
 'MYOD1_MOUSE',
 'P53_MOUSE',
 'PPARG_MOUSE',
 'PRGR_MOUSE',
 'REST_MOUSE',
 'RUNX1_MOUSE',
 'RXRA_MOUSE',
 'SOX2_MOUSE',
 'SPI1_MOUSE',
 'SRF_MOUSE',
 'STA5A_MOUSE',
 'STAT1_MOUSE',
 'STAT3_MOUSE',
 'TAL1_MOUSE',
 'TF65_MOUSE',
 'TFE2_MOUSE',
 'USF2_MOUSE']

In [ ]:
import matplotlib.ticker as ticker

count = []
TF_name = []

roc_auc_train_H_PWM_mono = []
roc_auc_test_H_PWM_mono = []
roc_auc_test_H_PWM = []
roc_auc_test_M_PWM = []
pr_auc_test_H_PWM = []
pr_auc_test_M_PWM = []
roc_auc_test_M_PWM_mono = []
roc_auc_train_H_PWM_di = []
roc_auc_test_H_PWM_di = []
roc_auc_test_M_PWM_di = []
roc_auc_test_M = []
mean_auc_test_M = []
median_auc_test_M = []
std_auc_test_M = []
roc_auc_test_H = []
mean_auc_test_H = []
median_auc_test_H = []
std_auc_test_H = []
roc_auc_train_H = []
mean_auc_train_H = []
median_auc_train_H = []
std_auc_train_H = []
pr_auc_train_H_PWM_mono = []
pr_auc_test_H_PWM_mono = []
pr_auc_test_M_PWM_mono = []
pr_auc_train_H_PWM_di = []
pr_auc_test_H_PWM_di = []
pr_auc_test_M_PWM_di = []
pr_auc_test_M = []
mean_pr_auc_test_M = []
median_pr_auc_test_M = []
std_pr_auc_test_M = []
pr_auc_test_H = []
mean_pr_auc_test_H = []
median_pr_auc_test_H = []
std_pr_auc_test_H = []
pr_auc_train_H = []
mean_pr_auc_train_H = []
median_pr_auc_train_H = []
std_pr_auc_train_H = []
Model_l = []
PWM = []



roc_auc2 = []
Model_l2 = []
PWM2 = []
hue2 = []
            
print("Sarus is running...")
print("selected_full_names_list:", len(selected_full_names_list))
calc = 0
success = 0

di_l = 0
di_n = ""

for global_index in range(len(selected_HUMAN)):
    for Model in ["RandomForestClassifier", "LogisticRegression", "XGBClassifier", "BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression"]: #"BaggingClassifier", 
        for pwm_name in ["mono", "di", "mono_di"]:
            print(Model, pwm_name)
            TF2 = sorted(selected_MOUSE)[global_index]
            TF1 = sorted(selected_HUMAN)[global_index]
            if TF1 in selected_HUMAN:
                print(TF1)
                calc += 1
                try:         
                    new_dir_name = outputdir + "/" + TF1.split("_")[0]
                    os.chdir(new_dir_name)

                    sns.set_context("paper", font_scale=3)

                    df = pd.read_csv(new_dir_name + f"/new_log_roc_pr_HUMAN_MOUSE_{pwm_name}_{Model}_{pwm_name}_15.04.2024.txt", sep=' ', index_col=False, header=0)
                    
                    data = list(df.features_c)

                    if 'features_c' in data:
                        last_feature_index = len(data) - 1 - data[::-1].index('features_c')
                        
                        extracted_data = data[last_feature_index + 1:]
                        
                        df = df.iloc[last_feature_index*-1:, :]


                    if pwm_name == "di":
                        names = [x.replace("dpwm", "PWM") for x in df.columns]
                        df.columns = names

                    if pwm_name == "mono" and Model == "RandomForestClassifier":
                        count.append(list(df["features_c"])[-1])
                        TF_name.append(TF1.split("_")[0])
                        roc_auc_train_H_PWM_mono.append(list(df["roc_auc_train_H_PWM_mono"])[-1])
                        roc_auc_test_H_PWM_mono.append(list(df["roc_auc_test_H_PWM_mono"])[-1])
                        roc_auc_test_H_PWM.append(list(df["roc_auc_test_H_PWM_mono"])[-1])
                        roc_auc_test_M_PWM.append(list(df["roc_auc_test_M_PWM_mono"])[-1])
                        pr_auc_test_H_PWM.append(list(df["pr_auc_test_H_PWM_mono"])[-1])
                        pr_auc_test_M_PWM.append(list(df["pr_auc_test_M_PWM_mono"])[-1])
                        
                        
                        roc_auc_test_M_PWM_mono.append(list(df["roc_auc_test_M_PWM_mono"])[-1])
                        roc_auc_train_H_PWM_di.append(0)
                        roc_auc_test_H_PWM_di.append(0)
                        roc_auc_test_M_PWM_di.append(0)
                        roc_auc_test_M.append(list(df["roc_auc_test_M_PWM_mono"])[-1])
                        #mean_auc_test_M.append(list(df["roc_auc_test_M_PWM_mono"])[-1])
                        #median_auc_test_M.append(list(df["roc_auc_test_M_PWM_mono"])[-1])
                        #std_auc_test_M.append(list(df["std_auc_test_M"])[-1])
                        roc_auc_test_H.append(list(df["roc_auc_test_H_PWM_mono"])[-1])
                        #mean_auc_test_H.append(list(df["roc_auc_test_H_PWM_mono"])[-1])
                        #median_auc_test_H.append(list(df["roc_auc_test_H_PWM_mono"])[-1])
                        #std_auc_test_H.append(list(df["std_auc_test_H"])[-1])
                        roc_auc_train_H.append(list(df["roc_auc_train_H_PWM_mono"])[-1])
                        #mean_auc_train_H.append(list(df["roc_auc_train_H_PWM_mono"])[-1])
                        #median_auc_train_H.append(list(df["roc_auc_train_H_PWM_mono"])[-1])
                        #std_auc_train_H.append(list(df["std_auc_train_H"])[-1])
                        pr_auc_train_H_PWM_mono.append(list(df["pr_auc_train_H_PWM_mono"])[-1])
                        pr_auc_test_H_PWM_mono.append(list(df["pr_auc_test_H_PWM_mono"])[-1])
                        pr_auc_test_M_PWM_mono.append(list(df["pr_auc_test_M_PWM_mono"])[-1])
                        pr_auc_train_H_PWM_di.append(0)
                        pr_auc_test_H_PWM_di.append(0)
                        pr_auc_test_M_PWM_di.append(0)
                        pr_auc_test_M.append(list(df["pr_auc_test_M_PWM_mono"])[-1])
                        #mean_pr_auc_test_M.append(list(df["pr_auc_test_M_PWM_mono"])[-1])
                        #median_pr_auc_test_M.append(list(df["pr_auc_test_M_PWM_mono"])[-1])
                        #std_pr_auc_test_M.append(list(df["std_pr_auc_test_M"])[-1])
                        pr_auc_test_H.append(list(df["pr_auc_test_H_PWM_mono"])[-1])
                        #mean_pr_auc_test_H.append(list(df["pr_auc_test_H_PWM_mono"])[-1])
                        #median_pr_auc_test_H.append(list(df["pr_auc_test_H_PWM_mono"])[-1])
                        #std_pr_auc_test_H.append(list(df["std_pr_auc_test_H"])[-1])
                        pr_auc_train_H.append(list(df["pr_auc_train_H_PWM_mono"])[-1])
                        #mean_pr_auc_train_H.append(list(df["pr_auc_train_H_PWM_mono"])[-1])
                        #median_pr_auc_train_H.append(list(df["pr_auc_train_H_PWM_mono"])[-1])
                        #std_pr_auc_train_H.append(list(df["std_pr_auc_train_H"])[-1])
                        Model_l.append("Single best mono PWM")
                        PWM.append("mono")


                    if pwm_name == "di" and Model == "RandomForestClassifier":
                        count.append(list(df["features_c"])[-1])
                        TF_name.append(TF1.split("_")[0])
                        roc_auc_train_H_PWM_mono.append(0)
                        roc_auc_test_H_PWM_mono.append(0)
                        roc_auc_test_M_PWM_mono.append(0)
                        roc_auc_train_H_PWM_di.append(list(df["roc_auc_train_H_PWM_di"])[-1])
                        roc_auc_test_H_PWM_di.append(list(df["roc_auc_test_H_PWM_di"])[-1])
                        roc_auc_test_H_PWM.append(list(df["roc_auc_test_H_PWM_di"])[-1])
                        roc_auc_test_M_PWM.append(list(df["roc_auc_test_M_PWM_di"])[-1])
                        pr_auc_test_H_PWM.append(list(df["pr_auc_test_H_PWM_di"])[-1])
                        pr_auc_test_M_PWM.append(list(df["pr_auc_test_M_PWM_di"])[-1])
                        
                        
                        roc_auc_test_M_PWM_di.append(list(df["roc_auc_test_M_PWM_di"])[-1])
                        roc_auc_test_M.append(list(df["roc_auc_test_M_PWM_di"])[-1])
                        #mean_auc_test_M.append(list(df["roc_auc_test_M_PWM_di"])[-1])
                        #median_auc_test_M.append(list(df["roc_auc_test_M_PWM_di"])[-1])
                        #std_auc_test_M.append(list(df["std_auc_test_M"])[-1])
                        roc_auc_test_H.append(list(df["roc_auc_test_H_PWM_di"])[-1])
                        #mean_auc_test_H.append(list(df["roc_auc_test_H_PWM_di"])[-1])
                        #median_auc_test_H.append(list(df["roc_auc_test_H_PWM_di"])[-1])
                        #std_auc_test_H.append(list(df["std_auc_test_H"])[-1])
                        roc_auc_train_H.append(list(df["roc_auc_train_H_PWM_di"])[-1])
                        #mean_auc_train_H.append(list(df["roc_auc_train_H_PWM_di"])[-1])
                        #median_auc_train_H.append(list(df["roc_auc_train_H_PWM_di"])[-1])
                        #std_auc_train_H.append(list(df["std_auc_train_H"])[-1])
                        pr_auc_train_H_PWM_mono.append(0)
                        pr_auc_test_H_PWM_mono.append(0)
                        pr_auc_test_M_PWM_mono.append(0)
                        pr_auc_train_H_PWM_di.append(list(df["pr_auc_train_H_PWM_di"])[-1])
                        pr_auc_test_H_PWM_di.append(list(df["pr_auc_test_H_PWM_di"])[-1])
                        pr_auc_test_M_PWM_di.append(list(df["pr_auc_test_M_PWM_di"])[-1])
                        pr_auc_test_M.append(list(df["pr_auc_test_M_PWM_di"])[-1])
                        #mean_pr_auc_test_M.append(list(df["pr_auc_test_M_PWM_di"])[-1])
                        #median_pr_auc_test_M.append(list(df["pr_auc_test_M_PWM_di"])[-1])
                        #std_pr_auc_test_M.append(list(df["std_pr_auc_test_M"])[-1])
                        pr_auc_test_H.append(list(df["pr_auc_test_H_PWM_di"])[-1])
                        #mean_pr_auc_test_H.append(list(df["pr_auc_test_H_PWM_di"])[-1])
                        #median_pr_auc_test_H.append(list(df["pr_auc_test_H_PWM_di"])[-1])
                        #std_pr_auc_test_H.append(list(df["std_pr_auc_test_H"])[-1])
                        pr_auc_train_H.append(list(df["pr_auc_train_H_PWM_di"])[-1])
                        #mean_pr_auc_train_H.append(list(df["pr_auc_train_H_PWM_di"])[-1])
                        #median_pr_auc_train_H.append(list(df["pr_auc_train_H_PWM_di"])[-1])
                        #std_pr_auc_train_H.append(list(df["std_pr_auc_train_H"])[-1])
                        Model_l.append("Single best di PWM")
                        PWM.append("di")

                    
                    if (pwm_name == "mono"):
                        mono_l = list(df["roc_auc_test_M_PWM_mono"])[-1]
                        mono_n = TF1.split("_")[0]
                        #print(mono_n)
                    if (pwm_name == "di"):
                        di_l = list(df["roc_auc_test_M_PWM_di"])[-1]
                        di_n = TF1.split("_")[0]
                        #print(mono_n)
                        


                    count.append(list(df["features_c"])[-1])
                    TF_name.append(TF1.split("_")[0])

                    
                    if (pwm_name == "mono_di")&(di_n == TF1.split("_")[0])&(mono_n==di_n):
                        roc_auc_test_M_PWM_mono.append(max(mono_l_m,di_l_m))
                        roc_auc_test_H_PWM_mono.append(max(mono_l_h,di_l_h))
                        pr_auc_test_M_PWM_mono.append(max(mono_l_m_pr,di_l_m_pr))
                        pr_auc_test_H_PWM_mono.append(max(mono_l_h_pr,di_l_h_pr))
                        roc_auc_test_H_PWM.append(max(mono_l_h,di_l_h))
                        roc_auc_test_M_PWM.append(max(mono_l_m,di_l_m))
                        pr_auc_test_H_PWM.append(max(mono_l_h_pr,di_l_h_pr))
                        pr_auc_test_M_PWM.append(max(mono_l_m_pr,di_l_m_pr))
                    
                        roc_auc_test_M_PWM_di.append(max(mono_l_m,di_l_m))
                        roc_auc_test_H_PWM_di.append(max(mono_l_h,di_l_h))
                        pr_auc_test_M_PWM_di.append(max(mono_l_m_pr,di_l_m_pr))
                        pr_auc_test_H_PWM_di.append(max(mono_l_h_pr,di_l_h_pr))
                        
                        
                        pr_auc_train_H_PWM_di.append(0)
                        pr_auc_train_H_PWM_mono.append(0)
                        roc_auc_train_H_PWM_mono.append(0)
                        roc_auc_train_H_PWM_di.append(0)
                        
                        
                    if (pwm_name == "di"):
                        
                        di_l_m = float(list(df["roc_auc_test_M_PWM_di"])[-1])
                        di_l_h = float(list(df["roc_auc_test_H_PWM_di"])[-1])
                        
                        di_l_m_pr = float(list(df["pr_auc_test_M_PWM_di"])[-1])
                        di_l_h_pr = float(list(df["pr_auc_test_H_PWM_di"])[-1])
                        
                        di_n = TF1.split("_")[0]
                        
                        
                        roc_auc_test_M_PWM_mono.append(list(df["roc_auc_test_M_PWM_di"])[-1])
                        roc_auc_test_H_PWM_mono.append(list(df["roc_auc_test_H_PWM_di"])[-1])
                        roc_auc_test_H_PWM.append(list(df["roc_auc_test_H_PWM_di"])[-1])
                        roc_auc_test_M_PWM.append(list(df["roc_auc_test_M_PWM_di"])[-1])
                        pr_auc_test_H_PWM.append(list(df["pr_auc_test_H_PWM_di"])[-1])
                        pr_auc_test_M_PWM.append(list(df["pr_auc_test_M_PWM_di"])[-1])
                        
                        pr_auc_train_H_PWM_di.append(list(df["pr_auc_train_H_PWM_di"])[-1])
                        pr_auc_test_H_PWM_di.append(list(df["pr_auc_test_H_PWM_di"])[-1])
                        pr_auc_test_M_PWM_di.append(list(df["pr_auc_test_M_PWM_di"])[-1])
                        pr_auc_train_H_PWM_mono.append(list(df["pr_auc_train_H_PWM_di"])[-1])
                        pr_auc_test_M_PWM_mono.append(list(df["pr_auc_test_M_PWM_di"])[-1])
                        pr_auc_test_H_PWM_mono.append(list(df["pr_auc_test_H_PWM_di"])[-1])
                        
                        roc_auc_train_H_PWM_mono.append(list(df["roc_auc_train_H_PWM_di"])[-1])
                        roc_auc_train_H_PWM_di.append(list(df["roc_auc_train_H_PWM_di"])[-1])
                        roc_auc_test_H_PWM_di.append(list(df["roc_auc_test_H_PWM_di"])[-1])
                        roc_auc_test_M_PWM_di.append(list(df["roc_auc_test_M_PWM_di"])[-1])
                    
                    if (pwm_name == "mono"):
                        
                        mono_l_m = float(list(df["roc_auc_test_M_PWM_mono"])[-1])
                        mono_l_h = float(list(df["roc_auc_test_H_PWM_mono"])[-1])
                        
                        mono_l_m_pr = float(list(df["pr_auc_test_M_PWM_mono"])[-1])
                        mono_l_h_pr = float(list(df["pr_auc_test_H_PWM_mono"])[-1])
                        
                        mono_n = TF1.split("_")[0]
                        
                        
                        roc_auc_test_M_PWM_mono.append(list(df["roc_auc_test_M_PWM_mono"])[-1])
                        roc_auc_test_H_PWM_mono.append(list(df["roc_auc_test_H_PWM_mono"])[-1])
                        roc_auc_test_H_PWM.append(list(df["roc_auc_test_H_PWM_mono"])[-1])
                        roc_auc_test_M_PWM.append(list(df["roc_auc_test_M_PWM_mono"])[-1])
                        pr_auc_test_H_PWM.append(list(df["pr_auc_test_H_PWM_mono"])[-1])
                        pr_auc_test_M_PWM.append(list(df["pr_auc_test_M_PWM_mono"])[-1])
                        
                        pr_auc_train_H_PWM_di.append(list(df["pr_auc_train_H_PWM_mono"])[-1])
                        pr_auc_test_H_PWM_di.append(list(df["pr_auc_test_H_PWM_mono"])[-1])
                        pr_auc_test_M_PWM_di.append(list(df["pr_auc_test_M_PWM_mono"])[-1])
                        pr_auc_train_H_PWM_mono.append(list(df["pr_auc_train_H_PWM_mono"])[-1])
                        pr_auc_test_M_PWM_mono.append(list(df["pr_auc_test_M_PWM_mono"])[-1])
                        pr_auc_test_H_PWM_mono.append(list(df["pr_auc_test_H_PWM_mono"])[-1])
                        
                        roc_auc_train_H_PWM_mono.append(list(df["roc_auc_train_H_PWM_mono"])[-1])
                        roc_auc_train_H_PWM_di.append(list(df["roc_auc_train_H_PWM_mono"])[-1])
                        roc_auc_test_H_PWM_di.append(list(df["roc_auc_test_H_PWM_mono"])[-1])
                        roc_auc_test_M_PWM_di.append(list(df["roc_auc_test_M_PWM_mono"])[-1])

                    roc_auc_test_M.append(list(df["roc_auc_test_M"])[-1])
                    #mean_auc_test_M.append(list(df["mean_auc_test_M"])[-1])
                    #median_auc_test_M.append(list(df["median_auc_test_M"])[-1])
                    #std_auc_test_M.append(list(df["std_auc_test_M"])[-1])
                    roc_auc_test_H.append(list(df["roc_auc_test_H"])[-1])
                    #mean_auc_test_H.append(list(df["mean_auc_test_H"])[-1])
                    #median_auc_test_H.append(list(df["median_auc_test_H"])[-1])
                    #std_auc_test_H.append(list(df["std_auc_test_H"])[-1])
                    roc_auc_train_H.append(list(df["roc_auc_train_H"])[-1])
                    #mean_auc_train_H.append(list(df["mean_auc_train_H"])[-1])
                    #median_auc_train_H.append(list(df["median_auc_train_H"])[-1])
                    #std_auc_train_H.append(list(df["std_auc_train_H"])[-1])
                    pr_auc_test_M.append(list(df["pr_auc_test_M"])[-1])
                    #mean_pr_auc_test_M.append(list(df["mean_pr_auc_test_M"])[-1])
                    #median_pr_auc_test_M.append(list(df["median_pr_auc_test_M"])[-1])
                    #std_pr_auc_test_M.append(list(df["std_pr_auc_test_M"])[-1])
                    pr_auc_test_H.append(list(df["pr_auc_test_H"])[-1])
                    #mean_pr_auc_test_H.append(list(df["mean_pr_auc_test_H"])[-1])
                    #median_pr_auc_test_H.append(list(df["median_pr_auc_test_H"])[-1])
                    #std_pr_auc_test_H.append(list(df["std_pr_auc_test_H"])[-1])
                    pr_auc_train_H.append(list(df["pr_auc_train_H"])[-1])
                    #mean_pr_auc_train_H.append(list(df["mean_pr_auc_train_H"])[-1])
                    #median_pr_auc_train_H.append(list(df["median_pr_auc_train_H"])[-1])
                    #std_pr_auc_train_H.append(list(df["std_pr_auc_train_H"])[-1])
                    Model_l.append(Model)
                    if (pwm_name == "mono_di"):
                        PWM.append("mono+di")
                    else:
                        PWM.append(pwm_name)



                except FileNotFoundError:
                    #print("FileNotFoundError")
                    plt.close('all')
                    #print("a")
                    continue

                  
df_total = pd.DataFrame({'Count':np.array(count, dtype=np.int32), "TF_name":TF_name, "Model":Model_l, "PWM":PWM,
                         "roc_auc_train_H_PWM_mono":np.array(roc_auc_train_H_PWM_mono, dtype=np.float64), 
                         "roc_auc_test_H_PWM_mono":np.array(roc_auc_test_H_PWM_mono, dtype=np.float64), 
                         "roc_auc_test_M_PWM_mono":np.array(roc_auc_test_M_PWM_mono, dtype=np.float64),
                         "roc_auc_train_H_PWM_di":np.array(roc_auc_train_H_PWM_di, dtype=np.float64), 
                         "roc_auc_test_H_PWM_di":np.array(roc_auc_test_H_PWM_di, dtype=np.float64), 
                         "roc_auc_test_M_PWM_di":np.array(roc_auc_test_M_PWM_di, dtype=np.float64),
                         "roc_auc_test_M":np.array(roc_auc_test_M, dtype=np.float64),
                         #"mean_auc_test_M":np.array(mean_auc_test_M, dtype=np.float64),
                         #"median_auc_test_M":np.array(median_auc_test_M, dtype=np.float64),
                         #"std_auc_test_M":np.array(std_auc_test_M, dtype=np.float64),
                         "roc_auc_test_H":np.array(roc_auc_test_H, dtype=np.float64),
                         #"mean_auc_test_H":np.array(mean_auc_test_H, dtype=np.float64),
                         #"median_auc_test_H":np.array(median_auc_test_H, dtype=np.float64),
                         #"std_auc_test_H":np.array(std_auc_test_H, dtype=np.float64), 
                         "roc_auc_train_H":np.array(roc_auc_train_H, dtype=np.float64),
                         #"mean_auc_train_H":np.array(mean_auc_train_H, dtype=np.float64),
                         #"median_auc_train_H":np.array(median_auc_train_H, dtype=np.float64),
                         #"std_auc_train_H":np.array(std_auc_train_H, dtype=np.float64),
                         "pr_auc_train_H_PWM_mono":np.array(pr_auc_train_H_PWM_mono, dtype=np.float64),
                         "pr_auc_test_H_PWM_mono":np.array(pr_auc_test_H_PWM_mono, dtype=np.float64),
                         "pr_auc_test_M_PWM_mono":np.array(pr_auc_test_M_PWM_mono, dtype=np.float64),
                         "pr_auc_train_H_PWM_di":np.array(pr_auc_train_H_PWM_di, dtype=np.float64),
                         "pr_auc_test_H_PWM_di":np.array(pr_auc_test_H_PWM_di, dtype=np.float64),
                         "pr_auc_test_M_PWM_di":np.array(pr_auc_test_M_PWM_di, dtype=np.float64),
                         "pr_auc_test_M":np.array(pr_auc_test_M, dtype=np.float64),
                         #"mean_pr_auc_test_M":np.array(mean_pr_auc_test_M, dtype=np.float64),
                         #"median_pr_auc_test_M":np.array(median_pr_auc_test_M, dtype=np.float64),
                         #"std_pr_auc_test_M":np.array(std_pr_auc_test_M, dtype=np.float64),
                         "pr_auc_test_H":np.array(pr_auc_test_H, dtype=np.float64),
                         #"mean_pr_auc_test_H":np.array(mean_pr_auc_test_H, dtype=np.float64),
                         #"median_pr_auc_test_H":np.array(median_pr_auc_test_H, dtype=np.float64),
                         #"std_pr_auc_test_H":np.array(std_pr_auc_test_H, dtype=np.float64),
                         "pr_auc_train_H":np.array(pr_auc_train_H, dtype=np.float64),
                         #"mean_pr_auc_train_H":np.array(mean_pr_auc_train_H, dtype=np.float64),
                         #"median_pr_auc_train_H":np.array(median_pr_auc_train_H, dtype=np.float64),
                         #"std_pr_auc_train_H":np.array(std_pr_auc_train_H, dtype=np.float64),
                         "roc_auc_test_H_PWM":np.array(roc_auc_test_H_PWM, dtype=np.float64),
                         "roc_auc_test_M_PWM":np.array(roc_auc_test_M_PWM, dtype=np.float64),
                         "pr_auc_test_H_PWM":np.array(pr_auc_test_H_PWM, dtype=np.float64),
                         "pr_auc_test_M_PWM":np.array(pr_auc_test_M_PWM, dtype=np.float64)})

df_total.to_csv("~/HUMAN_MOUSE_total_100k.csv", sep="\t", index=False)   


In [ ]:
ddf = pd.read_csv(root+"/TF-ML_last_v_22/df_Names_seq_list.csv",  sep='\t')
df_total = pd.merge(df_total, ddf, left_on='TF_name', right_on='Names', how='left')
df_total = df_total[df_total["Seq_count"] > 100]

In [ ]:
df_total=df_total.drop_duplicates() 

In [ ]:
df_total.to_csv("~/HUMAN_MOUSE_total_100k.csv", sep="\t", index=False)   

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
import seaborn as sns

sns.set(font_scale=4, style="ticks", font="Lato")
sns.set_style({"xtick.direction": "in","ytick.direction": "in"})
matplotlib.rcParams['font.weight'] = "medium"
matplotlib.rcParams['axes.labelweight'] = 'medium'
matplotlib.rcParams['figure.titleweight'] = 'medium'
matplotlib.rcParams['axes.titleweight'] = 'medium'
matplotlib.rcParams['figure.figsize'] = 20, 10

plt.title("", fontsize=30)

hue_order=["Single best mono PWM", "Single best di PWM", "LogisticRegression", "RandomForestClassifier", "XGBClassifier", "BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression"] #"BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression", "BaggingClassifier_RandomForestClassifier", "AdaBoostClassifier"]


ax = sns.swarmplot(x="PWM", y="roc_auc_test_M", hue="Model", order=["mono", "di", "mono+di"],
                   data=df_total, hue_order=hue_order,
              size=12, dodge=True, zorder=2, palette="Set2")

ax.axhline(np.median(df_total[df_total["Model"] == "Single best mono PWM"]["roc_auc_test_M"]), c="#66c2a5",ls="-.", linewidth=3, zorder=1)
ax.axhline(np.median(df_total[df_total["Model"] == "Single best di PWM"]["roc_auc_test_M"]), c="#fc8d62",ls="-.", linewidth=3, zorder=1) 

ax = sns.boxplot(x="PWM", y="roc_auc_test_M", hue="Model", data=df_total, hue_order=hue_order,
              fill=False, linewidth=3, gap=.2, zorder=3, color="black", showcaps=False, fliersize=0)

handles, labels = ax.get_legend_handles_labels()

l = plt.legend(handles[0:len(hue_order)], labels[0:len(hue_order)], bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

ax.set_xlabel('PWM type')
ax.set_ylabel('AUC ROC')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

plt.savefig(f"{os.path.join(cfg['paths']['output_dir'], '..', 'Models_comparison_ROC_H_M.pdf')}", dpi='figure', pad_inches=1, bbox_inches='tight', transparent=True)
plt.show()
plt.close('all')




In [ ]:
import seaborn as sns

sns.set(font_scale=4, style="ticks", font="Lato")
sns.set_style({"xtick.direction": "in","ytick.direction": "in"})
matplotlib.rcParams['font.weight'] = "medium"
matplotlib.rcParams['axes.labelweight'] = 'medium'
matplotlib.rcParams['figure.titleweight'] = 'medium'
matplotlib.rcParams['axes.titleweight'] = 'medium'
matplotlib.rcParams['figure.figsize'] = 20, 10

plt.title("", fontsize=30)

hue_order=["Single best mono PWM", "Single best di PWM", "LogisticRegression", "RandomForestClassifier", "XGBClassifier", "BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression"] #"BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression", "BaggingClassifier_RandomForestClassifier", "AdaBoostClassifier"]

ax = sns.swarmplot(x="PWM", y="pr_auc_test_M", hue="Model", order=["mono", "di", "mono+di"],
                   data=df_total, hue_order=hue_order,
              size=12, dodge=True, zorder=2, palette="Set2")

ax.axhline(np.median(df_total[df_total["Model"] == "Single best mono PWM"]["pr_auc_test_M"]), c="#66c2a5",ls="-.", linewidth=3, zorder=1)
ax.axhline(np.median(df_total[df_total["Model"] == "Single best di PWM"]["pr_auc_test_M"]), c="#fc8d62",ls="-.", linewidth=3, zorder=1) 

ax = sns.boxplot(x="PWM", y="pr_auc_test_M", hue="Model", data=df_total, hue_order=hue_order,
              fill=False, linewidth=3, gap=.2, zorder=3, color="black", showcaps=False, fliersize=0)

handles, labels = ax.get_legend_handles_labels()

l = plt.legend(handles[0:len(hue_order)], labels[0:len(hue_order)], bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

ax.set_xlabel('PWM type')
ax.set_ylabel('AUC PRC')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

plt.savefig(f"{os.path.join(cfg['paths']['output_dir'], '..', 'Models_comparison_PR_H_M.pdf')}", dpi='figure', pad_inches=1, bbox_inches='tight', transparent=True)
plt.show()
plt.close('all')




In [ ]:
import seaborn as sns

sns.set(font_scale=4, style="ticks", font="Lato")
sns.set_style({"xtick.direction": "in","ytick.direction": "in"})
matplotlib.rcParams['font.weight'] = "medium"
matplotlib.rcParams['axes.labelweight'] = 'medium'
matplotlib.rcParams['figure.titleweight'] = 'medium'
matplotlib.rcParams['axes.titleweight'] = 'medium'
matplotlib.rcParams['figure.figsize'] = 20, 10

plt.title("", fontsize=30)

hue_order=["Single best mono PWM", "Single best di PWM", "LogisticRegression", "RandomForestClassifier", "XGBClassifier", "BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression"] #"BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression", "BaggingClassifier_RandomForestClassifier", "AdaBoostClassifier"]


ax = sns.swarmplot(x="PWM", y="roc_auc_test_H", hue="Model", order=["mono", "di", "mono+di"],
                   data=df_total, hue_order=hue_order,
              size=12, dodge=True, zorder=2, palette="Set2")

ax.axhline(np.median(df_total[df_total["Model"] == "Single best mono PWM"]["roc_auc_test_H"]), c="#66c2a5",ls="-.", linewidth=3, zorder=1)
ax.axhline(np.median(df_total[df_total["Model"] == "Single best di PWM"]["roc_auc_test_H"]), c="#fc8d62",ls="-.", linewidth=3, zorder=1) 

ax = sns.boxplot(x="PWM", y="roc_auc_test_H", hue="Model", data=df_total, hue_order=hue_order,
              fill=False, linewidth=3, gap=.2, zorder=3, color="black", showcaps=False, fliersize=0)

handles, labels = ax.get_legend_handles_labels()

l = plt.legend(handles[0:len(hue_order)], labels[0:len(hue_order)], bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)


ax.set_xlabel('PWM type')
ax.set_ylabel('AUC ROC')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

plt.savefig(f"{os.path.join(cfg['paths']['output_dir'], '..', 'Models_comparison_ROC_H_H.pdf')}", dpi='figure', pad_inches=1, bbox_inches='tight', transparent=True)
plt.show()
plt.close('all')




In [ ]:
import seaborn as sns

sns.set(font_scale=4, style="ticks", font="Lato")
sns.set_style({"xtick.direction": "in","ytick.direction": "in"})
matplotlib.rcParams['font.weight'] = "medium"
matplotlib.rcParams['axes.labelweight'] = 'medium'
matplotlib.rcParams['figure.titleweight'] = 'medium'
matplotlib.rcParams['axes.titleweight'] = 'medium'
matplotlib.rcParams['figure.figsize'] = 20, 10

plt.title("", fontsize=30)

hue_order=["Single best mono PWM", "Single best di PWM", "LogisticRegression", "RandomForestClassifier", "XGBClassifier", "BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression"] #"BaggingClassifier_XGBClassifier", "BaggingClassifier_LogisticRegression", "BaggingClassifier_RandomForestClassifier", "AdaBoostClassifier"]

ax = sns.swarmplot(x="PWM", y="pr_auc_test_H", hue="Model", order=["mono", "di", "mono+di"],
                   data=df_total, hue_order=hue_order,
              size=12, dodge=True, zorder=2, palette="Set2")

ax.axhline(np.median(df_total[df_total["Model"] == "Single best mono PWM"]["pr_auc_test_H"]), c="#66c2a5",ls="-.", linewidth=3, zorder=1)
ax.axhline(np.median(df_total[df_total["Model"] == "Single best di PWM"]["pr_auc_test_H"]), c="#fc8d62",ls="-.", linewidth=3, zorder=1) 

ax = sns.boxplot(x="PWM", y="pr_auc_test_H", hue="Model", data=df_total, hue_order=hue_order,
              fill=False, linewidth=3, gap=.2, zorder=3, color="black", showcaps=False, fliersize=0)


handles, labels = ax.get_legend_handles_labels()

l = plt.legend(handles[0:len(hue_order)], labels[0:len(hue_order)], bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)


ax.set_xlabel('PWM type')
ax.set_ylabel('AUC PRC')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

plt.savefig(f"{os.path.join(cfg['paths']['output_dir'], '..', 'Models_comparison_PR_H_H.pdf')}", dpi='figure', pad_inches=1, bbox_inches='tight', transparent=True)
plt.show()
plt.close('all')

In [ ]:
# TODO add best mono PWM

In [ ]:
df_total_subset = df_total[(df_total.PWM == "mono")&(df_total.Model == "Single best mono PWM")][["TF_name", "roc_auc_train_H_PWM_mono", "roc_auc_test_H_PWM_mono", "roc_auc_test_M_PWM_mono"]]
df_total_subset = df_total_subset.drop_duplicates()

In [ ]:
SLIM_df = pd.read_csv(os.path.join(cfg['paths']['output_dir'], 'outputdir', 'HUMAN_MOUSE_SLIM_roc_mono_di_RandomForestClassifier.txt'), header=None, sep='\t')
SLIM_df.columns = ["TF_name", "Best PWM #", "Best PWM name", "Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  ", "Train SLIM m=0 HUMAN  ", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ","Train diChIPMunk HUMAN  ", "Test SLIM m=0 HUMAN  ", "Test SLIM m=1 HUMAN  ",  "Test SLIM m=-5 HUMAN  ", "Test diChIPMunk HUMAN  ", "Test SLIM m=0 MOUSE  ", "Test SLIM m=1 MOUSE  ",  "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train full RandomForest HUMAN  ", "Train RandomForest HUMAN  ", "Train RF+diChIPMunk HUMAN  ", "Train RF+SLIM m=1 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest HUMAN  ", "Test RandomForest HUMAN  ", "Test RF+diChIPMunk HUMAN  ", "Test RF+SLIM m=1 HUMAN  ", "Test RF+SLIM m=-5 HUMAN  ", "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM m=1 MOUSE  ", "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "]

SLIM_df.TF_name = SLIM_df.TF_name.str[:-6]

SLIM_df = pd.merge(SLIM_df, df_total_subset, on='TF_name')
SLIM_df = SLIM_df.drop(["Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  "], axis=1)
SLIM_df.columns = ["TF_name", "Best PWM #", "Best PWM name", "Train SLIM m=0 HUMAN  ", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ","Train diChIPMunk HUMAN  ", "Test SLIM m=0 HUMAN  ", "Test SLIM m=1 HUMAN  ",  "Test SLIM m=-5 HUMAN  ", "Test diChIPMunk HUMAN  ", "Test SLIM m=0 MOUSE  ", "Test SLIM m=1 MOUSE  ",  "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train full RandomForest HUMAN  ", "Train RandomForest HUMAN  ", "Train RF+diChIPMunk HUMAN  ", "Train RF+SLIM m=1 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest HUMAN  ", "Test RandomForest HUMAN  ", "Test RF+diChIPMunk HUMAN  ", "Test RF+SLIM m=1 HUMAN  ", "Test RF+SLIM m=-5 HUMAN  ", "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM m=1 MOUSE  ", "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  ", "Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  "]
SLIM_df

In [ ]:
#SLIM_df = pd.read_csv(outputdir + "/" + "HUMAN_MOUSE_SLIM.txt", header=None, sep='\t')

# SLIM_df = pd.read_csv(os.path.join(cfg['paths']['output_dir'], 'outputdir', 'HUMAN_MOUSE_SLIM_roc_mono_di_RandomForestClassifier.txt'), header=None, sep='\t')
# #SLIM_df.columns = ["TF_name", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ", "Train diChIPMunk HUMAN  ", "Test SLIM m=1 MOUSE  ", "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM 1 MOUSE  ", "Test RF+SLIM -5 MOUSE  ", "Test RF+SLIM 1,-5 +diChIPMunk MOUSE  "]
# SLIM_df.columns = ["TF_name", "Best PWM #", "Best PWM name", "Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  ", "Train SLIM m=0 HUMAN  ", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ","Train diChIPMunk HUMAN  ", "Test SLIM m=0 HUMAN  ", "Test SLIM m=1 HUMAN  ",  "Test SLIM m=-5 HUMAN  ", "Test diChIPMunk HUMAN  ", "Test SLIM m=0 MOUSE  ", "Test SLIM m=1 MOUSE  ",  "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train full RandomForest HUMAN  ", "Train RandomForest HUMAN  ", "Train RF+diChIPMunk HUMAN  ", "Train RF+SLIM m=1 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest HUMAN  ", "Test RandomForest HUMAN  ", "Test RF+diChIPMunk HUMAN  ", "Test RF+SLIM m=1 HUMAN  ", "Test RF+SLIM m=-5 HUMAN  ", "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM m=1 MOUSE  ", "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "]

SLIM_df1 = pd.DataFrame()
#SLIM_df1["Train SLIM m=0 HUMAN  "] = SLIM_df["Train SLIM m=0 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=1 HUMAN  "] = SLIM_df["Train SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-5 HUMAN  "] = SLIM_df["Train SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-7 HUMAN  "] = SLIM_df["Train SLIM m=-7 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train diChIPMunk HUMAN  "] = SLIM_df["Train diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
SLIM_df1["Test SLIM m=0 HUMAN  "] = SLIM_df["Test SLIM m=0 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test SLIM m=1 HUMAN  "] = SLIM_df["Test SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test SLIM m=-5 HUMAN  "] = SLIM_df["Test SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=-7 HUMAN  "] = SLIM_df["Test SLIM m=-7 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test diChIPMunk HUMAN  "] = SLIM_df["Test diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=0 MOUSE  "] = SLIM_df["Test SLIM m=0 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=1 MOUSE  "] = SLIM_df["Test SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=-5 MOUSE  "] = SLIM_df["Test SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=-7 MOUSE  "] = SLIM_df["Test SLIM m=-7 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test diChIPMunk MOUSE  "] = SLIM_df["Test diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Train full RandomForest HUMAN  "] = SLIM_df["Train full RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RandomForest HUMAN  "] = SLIM_df["Train RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+diChIPMunk HUMAN  "] = SLIM_df["Train RF+diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1 HUMAN  "] = SLIM_df["Train RF+SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=-5 HUMAN  "] = SLIM_df["Train RF+SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
SLIM_df1["Test RandomForest HUMAN  "] = SLIM_df["Test RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+diChIPMunk HUMAN  "] = SLIM_df["Test RF+diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+SLIM m=1 HUMAN  "] = SLIM_df["Test RF+SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+SLIM m=-5 HUMAN  "] = SLIM_df["Test RF+SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test full RandomForest HUMAN  "] = SLIM_df["Test full RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test full RandomForest MOUSE  "] = SLIM_df["Test full RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RandomForest MOUSE  "] = SLIM_df["Test RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+diChIPMunk MOUSE  "] = SLIM_df["Test RF+diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+SLIM m=1 MOUSE  "] = SLIM_df["Test RF+SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+SLIM m=-5 MOUSE  "] = SLIM_df["Test RF+SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df=SLIM_df.drop(["Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test PWM HUMAN  ", 
#                      "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  "], axis=1)

SLIM_df1["TF_name"] = SLIM_df["TF_name"]
df_melted = pd.melt(SLIM_df1, id_vars=["TF_name"])


qualitative_colors = sns.color_palette("Set3", 7)
qualitative_colors
qualitative_colors_index = [0,0,0,1,1,1,2,2,2,3,3,3,3,3,4,4,4,4,4,5,5,5,5,5]
print(len(qualitative_colors_index))

sns.set(font_scale=4, style="ticks", font="Lato")
sns.set_style({"xtick.direction": "in","ytick.direction": "in"})
matplotlib.rcParams['font.weight'] = "medium"
matplotlib.rcParams['axes.labelweight'] = 'medium'
matplotlib.rcParams['figure.titleweight'] = 'medium'
matplotlib.rcParams['axes.titleweight'] = 'medium'
matplotlib.rcParams['figure.figsize'] = 10, 20

plt.title("", fontsize=30)

my_pal = {#"Train SLIM m=0 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=1 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=-5 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=-7 HUMAN  ":qualitative_colors[0], 
          #"Train diChIPMunk HUMAN  ":qualitative_colors[0],
          "Test SLIM m=0 HUMAN  ":qualitative_colors[1], 
          "Test SLIM m=1 HUMAN  ":qualitative_colors[1], 
          "Test SLIM m=-5 HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=-7 HUMAN  ":qualitative_colors[1], 
          "Test diChIPMunk HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=0 MOUSE  ":qualitative_colors[2], 
          #"Test SLIM m=1 MOUSE  ":qualitative_colors[2], 
          #"Test SLIM m=-5 MOUSE  ":qualitative_colors[2], 
          #"Test SLIM m=-7 MOUSE  ":qualitative_colors[2], 
          #"Test diChIPMunk MOUSE  ":qualitative_colors[2],
          #"Train full RandomForest HUMAN  ":qualitative_colors[3], 
          #"Train RandomForest HUMAN  ":qualitative_colors[3], 
          #"Train RF+diChIPMunk HUMAN  ":qualitative_colors[3], 
          #"Train RF+SLIM m=1 HUMAN  ":qualitative_colors[3],
          #"Train RF+SLIM m=-5 HUMAN  ":qualitative_colors[3],
          #"Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ":qualitative_colors[3],
          "Test RandomForest HUMAN  ":qualitative_colors[4], 
          "Test RF+diChIPMunk HUMAN  ":qualitative_colors[4], 
          "Test RF+SLIM m=1 HUMAN  ":qualitative_colors[4],
          "Test RF+SLIM m=-5 HUMAN  ":qualitative_colors[4],
          "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ":qualitative_colors[4],
          "Test full RandomForest HUMAN  ":qualitative_colors[4], 
          #"Test full RandomForest MOUSE  ":qualitative_colors[5], 
          #"Test RandomForest MOUSE  ":qualitative_colors[5], 
          #"Test RF+diChIPMunk MOUSE  ":qualitative_colors[5], 
          #"Test RF+SLIM m=1 MOUSE  ":qualitative_colors[5],
          #"Test RF+SLIM m=-5 MOUSE  ":qualitative_colors[5],
          #"Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  ":qualitative_colors[5]
}


ax = sns.swarmplot(x="value", y="variable",
                   data=df_melted,
                   size=7, zorder=2, palette="tab10")

ax = sns.boxplot(x="value", y="variable", 
                 data=df_melted,
                 fill=False, linewidth=3, zorder=3, color="black", showcaps=False, fliersize=0)



results = df_melted.groupby('variable').apply(lambda group: wilcoxon(group['value'], alternative='greater')).reset_index()
results.columns = ['variable', 'wilcoxon_result']

results['statistic'], results['p_value'] = zip(*results['wilcoxon_result'])

df = df_melted
i=0
for variable in [x[0] for x in list(my_pal.items())]:
    p_value = float(results[results['variable'] == variable]['p_value'])
    y = df[df['variable'] == variable]['value'].max() + 0.02 
    print(variable, p_value)
    if p_value < 0.001:
        plt.text(y, i, '***', ha='center', color='red')
    elif p_value < 0.01:
        plt.text(y, i, '**', ha='center', color='red')
    elif p_value < 0.05:
        plt.text(y, i, '*', ha='center', color='red')
    i+=1


ax.set_ylabel('Model')
ax.set_xlabel('Δ  AUC ROC')
ax.axvline(0, ls='-.', linewidth=3, zorder=1, color="black")
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

plt.savefig(f"{os.path.join(cfg['paths']['output_dir'], '..', 'Overall_improvement_HUMAN_ROC_all.pdf')}", dpi='figure', pad_inches=1, bbox_inches='tight', transparent=True)


In [ ]:
#SLIM_df = pd.read_csv(outputdir + "/" + "HUMAN_MOUSE_SLIM.txt", header=None, sep='\t')

# SLIM_df = pd.read_csv(os.path.join(cfg['paths']['output_dir'], 'outputdir', 'HUMAN_MOUSE_SLIM_roc_mono_di_RandomForestClassifier.txt'), header=None, sep='\t')
# #SLIM_df.columns = ["TF_name", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ", "Train diChIPMunk HUMAN  ", "Test SLIM m=1 MOUSE  ", "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM 1 MOUSE  ", "Test RF+SLIM -5 MOUSE  ", "Test RF+SLIM 1,-5 +diChIPMunk MOUSE  "]
# SLIM_df.columns = ["TF_name", "Best PWM #", "Best PWM name", "Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  ", "Train SLIM m=0 HUMAN  ", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ","Train diChIPMunk HUMAN  ", "Test SLIM m=0 HUMAN  ", "Test SLIM m=1 HUMAN  ",  "Test SLIM m=-5 HUMAN  ", "Test diChIPMunk HUMAN  ", "Test SLIM m=0 MOUSE  ", "Test SLIM m=1 MOUSE  ",  "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train full RandomForest HUMAN  ", "Train RandomForest HUMAN  ", "Train RF+diChIPMunk HUMAN  ", "Train RF+SLIM m=1 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest HUMAN  ", "Test RandomForest HUMAN  ", "Test RF+diChIPMunk HUMAN  ", "Test RF+SLIM m=1 HUMAN  ", "Test RF+SLIM m=-5 HUMAN  ", "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM m=1 MOUSE  ", "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "]

SLIM_df1 = pd.DataFrame()
#SLIM_df1["Train SLIM m=0 HUMAN  "] = SLIM_df["Train SLIM m=0 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=1 HUMAN  "] = SLIM_df["Train SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-5 HUMAN  "] = SLIM_df["Train SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-7 HUMAN  "] = SLIM_df["Train SLIM m=-7 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train diChIPMunk HUMAN  "] = SLIM_df["Train diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Test SLIM m=0 HUMAN  "] = SLIM_df["Test SLIM m=0 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=1 HUMAN  "] = SLIM_df["Test SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=-5 HUMAN  "] = SLIM_df["Test SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=-7 HUMAN  "] = SLIM_df["Test SLIM m=-7 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test diChIPMunk HUMAN  "] = SLIM_df["Test diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test SLIM m=0 MOUSE  "] = SLIM_df["Test SLIM m=0 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test SLIM m=1 MOUSE  "] = SLIM_df["Test SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test SLIM m=-5 MOUSE  "] = SLIM_df["Test SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=-7 MOUSE  "] = SLIM_df["Test SLIM m=-7 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test diChIPMunk MOUSE  "] = SLIM_df["Test diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Train full RandomForest HUMAN  "] = SLIM_df["Train full RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RandomForest HUMAN  "] = SLIM_df["Train RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+diChIPMunk HUMAN  "] = SLIM_df["Train RF+diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1 HUMAN  "] = SLIM_df["Train RF+SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=-5 HUMAN  "] = SLIM_df["Train RF+SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Test full RandomForest HUMAN  "] = SLIM_df["Test full RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RandomForest HUMAN  "] = SLIM_df["Test RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RF+diChIPMunk HUMAN  "] = SLIM_df["Test RF+diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RF+SLIM m=1 HUMAN  "] = SLIM_df["Test RF+SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RF+SLIM m=-5 HUMAN  "] = SLIM_df["Test RF+SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test full RandomForest MOUSE  "] = SLIM_df["Test full RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test RandomForest MOUSE  "] = SLIM_df["Test RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test RF+diChIPMunk MOUSE  "] = SLIM_df["Test RF+diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test RF+SLIM m=1 MOUSE  "] = SLIM_df["Test RF+SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test RF+SLIM m=-5 MOUSE  "] = SLIM_df["Test RF+SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df=SLIM_df.drop(["Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test PWM HUMAN  ", 
#                      "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  "], axis=1)

SLIM_df1["TF_name"] = SLIM_df["TF_name"]
df_melted = pd.melt(SLIM_df1, id_vars=["TF_name"])

import seaborn as sns


qualitative_colors = sns.color_palette("Set3", 7)
qualitative_colors
qualitative_colors_index = [0,0,0,1,1,1,2,2,2,3,3,3,3,3,4,4,4,4,4,5,5,5,5,5]
print(len(qualitative_colors_index))

sns.set(font_scale=4, style="ticks", font="Lato")
sns.set_style({"xtick.direction": "in","ytick.direction": "in"})
matplotlib.rcParams['font.weight'] = "medium"
matplotlib.rcParams['axes.labelweight'] = 'medium'
matplotlib.rcParams['figure.titleweight'] = 'medium'
matplotlib.rcParams['axes.titleweight'] = 'medium'
matplotlib.rcParams['figure.figsize'] = 10, 20

plt.title("", fontsize=30)

my_pal = {#"Train SLIM m=0 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=1 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=-5 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=-7 HUMAN  ":qualitative_colors[0], 
          #"Train diChIPMunk HUMAN  ":qualitative_colors[0],
          #"Test SLIM m=0 HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=1 HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=-5 HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=-7 HUMAN  ":qualitative_colors[1], 
          #"Test diChIPMunk HUMAN  ":qualitative_colors[1], 
          "Test SLIM m=0 MOUSE  ":qualitative_colors[2], 
          "Test SLIM m=1 MOUSE  ":qualitative_colors[2], 
          "Test SLIM m=-5 MOUSE  ":qualitative_colors[2], 
          #"Test SLIM m=-7 MOUSE  ":qualitative_colors[2], 
          "Test diChIPMunk MOUSE  ":qualitative_colors[2],
          #"Train full RandomForest HUMAN  ":qualitative_colors[3], 
          #"Train RandomForest HUMAN  ":qualitative_colors[3], 
          #"Train RF+diChIPMunk HUMAN  ":qualitative_colors[3], 
          #"Train RF+SLIM m=1 HUMAN  ":qualitative_colors[3],
          #"Train RF+SLIM m=-5 HUMAN  ":qualitative_colors[3],
          #"Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ":qualitative_colors[3],
          #"Test full RandomForest HUMAN  ":qualitative_colors[4], 
          #"Test RandomForest HUMAN  ":qualitative_colors[4], 
          #"Test RF+diChIPMunk HUMAN  ":qualitative_colors[4], 
          #"Test RF+SLIM m=1 HUMAN  ":qualitative_colors[4],
          #"Test RF+SLIM m=-5 HUMAN  ":qualitative_colors[4],
          #"Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ":qualitative_colors[4],
          "Test full RandomForest MOUSE  ":qualitative_colors[5], 
          "Test RandomForest MOUSE  ":qualitative_colors[5], 
          "Test RF+diChIPMunk MOUSE  ":qualitative_colors[5], 
          "Test RF+SLIM m=1 MOUSE  ":qualitative_colors[5],
          "Test RF+SLIM m=-5 MOUSE  ":qualitative_colors[5],
          "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  ":qualitative_colors[5]}


print(my_pal)
ax = sns.swarmplot(x="value", y="variable",
                   data=df_melted,
                   size=7, zorder=2, palette="tab10")

ax = sns.boxplot(x="value", y="variable", 
                 data=df_melted,
                 fill=False, linewidth=3, zorder=3, color="black", showcaps=False, fliersize=0)



results = df_melted.groupby('variable').apply(lambda group: wilcoxon(group['value'], alternative='greater')).reset_index()
results.columns = ['variable', 'wilcoxon_result']
results['statistic'], results['p_value'] = zip(*results['wilcoxon_result'])
df = df_melted
i=0
for variable in [x[0] for x in list(my_pal.items())]:
    p_value = float(results[results['variable'] == variable]['p_value'])
    y = df[df['variable'] == variable]['value'].max() + 0.02
    print(variable, p_value)
    if p_value < 0.001:
        plt.text(y, i, '***', ha='center', color='red')
    elif p_value < 0.01:
        plt.text(y, i, '**', ha='center', color='red')
    elif p_value < 0.05:
        plt.text(y, i, '*', ha='center', color='red')
    i+=1

ax.set_ylabel('Model')
ax.set_xlabel('Δ  AUC ROC')
ax.axvline(0, ls='-.', linewidth=3, zorder=1, color="black")
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

plt.savefig(f"{os.path.join(cfg['paths']['output_dir'], '..', 'Overall_improvement_MOUSE_ROC_all.pdf')}", dpi='figure', pad_inches=1, bbox_inches='tight', transparent=True)

In [ ]:
df_total_subset = df_total[(df_total.PWM == "mono")&(df_total.Model == "Single best mono PWM")][["TF_name", "pr_auc_train_H_PWM_mono", "pr_auc_test_H_PWM_mono", "pr_auc_test_M_PWM_mono"]]
df_total_subset = df_total_subset.drop_duplicates()

In [ ]:
SLIM_df = pd.read_csv(os.path.join(cfg['paths']['output_dir'], 'outputdir', 'HUMAN_MOUSE_SLIM_pr_mono_di_RandomForestClassifier.txt'), header=None, sep='\t')
SLIM_df.columns = ["TF_name", "Best PWM #", "Best PWM name", "Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  ", "Train SLIM m=0 HUMAN  ", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ","Train diChIPMunk HUMAN  ", "Test SLIM m=0 HUMAN  ", "Test SLIM m=1 HUMAN  ",  "Test SLIM m=-5 HUMAN  ", "Test diChIPMunk HUMAN  ", "Test SLIM m=0 MOUSE  ", "Test SLIM m=1 MOUSE  ",  "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train full RandomForest HUMAN  ", "Train RandomForest HUMAN  ", "Train RF+diChIPMunk HUMAN  ", "Train RF+SLIM m=1 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest HUMAN  ", "Test RandomForest HUMAN  ", "Test RF+diChIPMunk HUMAN  ", "Test RF+SLIM m=1 HUMAN  ", "Test RF+SLIM m=-5 HUMAN  ", "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM m=1 MOUSE  ", "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "]
SLIM_df.TF_name = SLIM_df.TF_name.str[:-6]
SLIM_df = pd.merge(SLIM_df, df_total_subset, on='TF_name')
SLIM_df = SLIM_df.drop(["Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  "], axis=1)
SLIM_df.columns = ["TF_name", "Best PWM #", "Best PWM name", "Train SLIM m=0 HUMAN  ", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ","Train diChIPMunk HUMAN  ", "Test SLIM m=0 HUMAN  ", "Test SLIM m=1 HUMAN  ",  "Test SLIM m=-5 HUMAN  ", "Test diChIPMunk HUMAN  ", "Test SLIM m=0 MOUSE  ", "Test SLIM m=1 MOUSE  ",  "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train full RandomForest HUMAN  ", "Train RandomForest HUMAN  ", "Train RF+diChIPMunk HUMAN  ", "Train RF+SLIM m=1 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest HUMAN  ", "Test RandomForest HUMAN  ", "Test RF+diChIPMunk HUMAN  ", "Test RF+SLIM m=1 HUMAN  ", "Test RF+SLIM m=-5 HUMAN  ", "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM m=1 MOUSE  ", "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  ", "Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  "]

In [ ]:
#SLIM_df = pd.read_csv(outputdir + "/" + "HUMAN_MOUSE_SLIM.txt", header=None, sep='\t')

# SLIM_df = pd.read_csv(os.path.join(cfg['paths']['output_dir'], 'outputdir', 'HUMAN_MOUSE_SLIM_roc_mono_di_RandomForestClassifier.txt'), header=None, sep='\t')
# #SLIM_df.columns = ["TF_name", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ", "Train diChIPMunk HUMAN  ", "Test SLIM m=1 MOUSE  ", "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM 1 MOUSE  ", "Test RF+SLIM -5 MOUSE  ", "Test RF+SLIM 1,-5 +diChIPMunk MOUSE  "]
# SLIM_df.columns = ["TF_name", "Best PWM #", "Best PWM name", "Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  ", "Train SLIM m=0 HUMAN  ", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ","Train diChIPMunk HUMAN  ", "Test SLIM m=0 HUMAN  ", "Test SLIM m=1 HUMAN  ",  "Test SLIM m=-5 HUMAN  ", "Test diChIPMunk HUMAN  ", "Test SLIM m=0 MOUSE  ", "Test SLIM m=1 MOUSE  ",  "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train full RandomForest HUMAN  ", "Train RandomForest HUMAN  ", "Train RF+diChIPMunk HUMAN  ", "Train RF+SLIM m=1 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest HUMAN  ", "Test RandomForest HUMAN  ", "Test RF+diChIPMunk HUMAN  ", "Test RF+SLIM m=1 HUMAN  ", "Test RF+SLIM m=-5 HUMAN  ", "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM m=1 MOUSE  ", "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "]

SLIM_df1 = pd.DataFrame()
#SLIM_df1["Train SLIM m=0 HUMAN  "] = SLIM_df["Train SLIM m=0 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=1 HUMAN  "] = SLIM_df["Train SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-5 HUMAN  "] = SLIM_df["Train SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-7 HUMAN  "] = SLIM_df["Train SLIM m=-7 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train diChIPMunk HUMAN  "] = SLIM_df["Train diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
SLIM_df1["Test SLIM m=0 HUMAN  "] = SLIM_df["Test SLIM m=0 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test SLIM m=1 HUMAN  "] = SLIM_df["Test SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test SLIM m=-5 HUMAN  "] = SLIM_df["Test SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=-7 HUMAN  "] = SLIM_df["Test SLIM m=-7 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test diChIPMunk HUMAN  "] = SLIM_df["Test diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=0 MOUSE  "] = SLIM_df["Test SLIM m=0 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=1 MOUSE  "] = SLIM_df["Test SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=-5 MOUSE  "] = SLIM_df["Test SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=-7 MOUSE  "] = SLIM_df["Test SLIM m=-7 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test diChIPMunk MOUSE  "] = SLIM_df["Test diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Train full RandomForest HUMAN  "] = SLIM_df["Train full RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RandomForest HUMAN  "] = SLIM_df["Train RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+diChIPMunk HUMAN  "] = SLIM_df["Train RF+diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1 HUMAN  "] = SLIM_df["Train RF+SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=-5 HUMAN  "] = SLIM_df["Train RF+SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
SLIM_df1["Test RandomForest HUMAN  "] = SLIM_df["Test RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+diChIPMunk HUMAN  "] = SLIM_df["Test RF+diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+SLIM m=1 HUMAN  "] = SLIM_df["Test RF+SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+SLIM m=-5 HUMAN  "] = SLIM_df["Test RF+SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test full RandomForest HUMAN  "] = SLIM_df["Test full RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test full RandomForest MOUSE  "] = SLIM_df["Test full RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RandomForest MOUSE  "] = SLIM_df["Test RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+diChIPMunk MOUSE  "] = SLIM_df["Test RF+diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+SLIM m=1 MOUSE  "] = SLIM_df["Test RF+SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+SLIM m=-5 MOUSE  "] = SLIM_df["Test RF+SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df=SLIM_df.drop(["Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test PWM HUMAN  ", 
#                      "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  "], axis=1)

SLIM_df1["TF_name"] = SLIM_df["TF_name"]
df_melted = pd.melt(SLIM_df1, id_vars=["TF_name"])
df_melted

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

grouped = df_melted.groupby('variable')

def perform_wilcoxon_test(group):
    stat, p_value = wilcoxon(group['value'], alternative='greater')
    return pd.Series({'statistic': stat, 'p_value': p_value})

results = grouped.apply(perform_wilcoxon_test).reset_index()

print(results)

In [ ]:
#SLIM_df = pd.read_csv(outputdir + "/" + "HUMAN_MOUSE_SLIM.txt", header=None, sep='\t')

# SLIM_df = pd.read_csv(os.path.join(cfg['paths']['output_dir'], 'outputdir', 'HUMAN_MOUSE_SLIM_roc_mono_di_RandomForestClassifier.txt'), header=None, sep='\t')
# #SLIM_df.columns = ["TF_name", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ", "Train diChIPMunk HUMAN  ", "Test SLIM m=1 MOUSE  ", "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM 1 MOUSE  ", "Test RF+SLIM -5 MOUSE  ", "Test RF+SLIM 1,-5 +diChIPMunk MOUSE  "]
# SLIM_df.columns = ["TF_name", "Best PWM #", "Best PWM name", "Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  ", "Train SLIM m=0 HUMAN  ", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ","Train diChIPMunk HUMAN  ", "Test SLIM m=0 HUMAN  ", "Test SLIM m=1 HUMAN  ",  "Test SLIM m=-5 HUMAN  ", "Test diChIPMunk HUMAN  ", "Test SLIM m=0 MOUSE  ", "Test SLIM m=1 MOUSE  ",  "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train full RandomForest HUMAN  ", "Train RandomForest HUMAN  ", "Train RF+diChIPMunk HUMAN  ", "Train RF+SLIM m=1 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest HUMAN  ", "Test RandomForest HUMAN  ", "Test RF+diChIPMunk HUMAN  ", "Test RF+SLIM m=1 HUMAN  ", "Test RF+SLIM m=-5 HUMAN  ", "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM m=1 MOUSE  ", "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "]

SLIM_df1 = pd.DataFrame()
#SLIM_df1["Train SLIM m=0 HUMAN  "] = SLIM_df["Train SLIM m=0 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=1 HUMAN  "] = SLIM_df["Train SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-5 HUMAN  "] = SLIM_df["Train SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-7 HUMAN  "] = SLIM_df["Train SLIM m=-7 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train diChIPMunk HUMAN  "] = SLIM_df["Train diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
SLIM_df1["Test SLIM m=0 HUMAN  "] = SLIM_df["Test SLIM m=0 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test SLIM m=1 HUMAN  "] = SLIM_df["Test SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test SLIM m=-5 HUMAN  "] = SLIM_df["Test SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=-7 HUMAN  "] = SLIM_df["Test SLIM m=-7 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test diChIPMunk HUMAN  "] = SLIM_df["Test diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=0 MOUSE  "] = SLIM_df["Test SLIM m=0 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=1 MOUSE  "] = SLIM_df["Test SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=-5 MOUSE  "] = SLIM_df["Test SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=-7 MOUSE  "] = SLIM_df["Test SLIM m=-7 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test diChIPMunk MOUSE  "] = SLIM_df["Test diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Train full RandomForest HUMAN  "] = SLIM_df["Train full RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RandomForest HUMAN  "] = SLIM_df["Train RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+diChIPMunk HUMAN  "] = SLIM_df["Train RF+diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1 HUMAN  "] = SLIM_df["Train RF+SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=-5 HUMAN  "] = SLIM_df["Train RF+SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
SLIM_df1["Test RandomForest HUMAN  "] = SLIM_df["Test RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+diChIPMunk HUMAN  "] = SLIM_df["Test RF+diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+SLIM m=1 HUMAN  "] = SLIM_df["Test RF+SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+SLIM m=-5 HUMAN  "] = SLIM_df["Test RF+SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test full RandomForest HUMAN  "] = SLIM_df["Test full RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test full RandomForest MOUSE  "] = SLIM_df["Test full RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RandomForest MOUSE  "] = SLIM_df["Test RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+diChIPMunk MOUSE  "] = SLIM_df["Test RF+diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+SLIM m=1 MOUSE  "] = SLIM_df["Test RF+SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+SLIM m=-5 MOUSE  "] = SLIM_df["Test RF+SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df=SLIM_df.drop(["Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test PWM HUMAN  ", 
#                      "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  "], axis=1)

SLIM_df1["TF_name"] = SLIM_df["TF_name"]
df_melted = pd.melt(SLIM_df1, id_vars=["TF_name"])


qualitative_colors = sns.color_palette("Set3", 7)
qualitative_colors
qualitative_colors_index = [0,0,0,1,1,1,2,2,2,3,3,3,3,3,4,4,4,4,4,5,5,5,5,5]
print(len(qualitative_colors_index))

sns.set(font_scale=4, style="ticks", font="Lato")
sns.set_style({"xtick.direction": "in","ytick.direction": "in"})
matplotlib.rcParams['font.weight'] = "medium"
matplotlib.rcParams['axes.labelweight'] = 'medium'
matplotlib.rcParams['figure.titleweight'] = 'medium'
matplotlib.rcParams['axes.titleweight'] = 'medium'
matplotlib.rcParams['figure.figsize'] = 10, 20

plt.title("", fontsize=30)

my_pal = {#"Train SLIM m=0 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=1 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=-5 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=-7 HUMAN  ":qualitative_colors[0], 
          #"Train diChIPMunk HUMAN  ":qualitative_colors[0],
          "Test SLIM m=0 HUMAN  ":qualitative_colors[1], 
          "Test SLIM m=1 HUMAN  ":qualitative_colors[1], 
          "Test SLIM m=-5 HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=-7 HUMAN  ":qualitative_colors[1], 
          "Test diChIPMunk HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=0 MOUSE  ":qualitative_colors[2], 
          #"Test SLIM m=1 MOUSE  ":qualitative_colors[2], 
          #"Test SLIM m=-5 MOUSE  ":qualitative_colors[2], 
          #"Test SLIM m=-7 MOUSE  ":qualitative_colors[2], 
          #"Test diChIPMunk MOUSE  ":qualitative_colors[2],
          #"Train full RandomForest HUMAN  ":qualitative_colors[3], 
          #"Train RandomForest HUMAN  ":qualitative_colors[3], 
          #"Train RF+diChIPMunk HUMAN  ":qualitative_colors[3], 
          #"Train RF+SLIM m=1 HUMAN  ":qualitative_colors[3],
          #"Train RF+SLIM m=-5 HUMAN  ":qualitative_colors[3],
          #"Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ":qualitative_colors[3],
          "Test RandomForest HUMAN  ":qualitative_colors[4], 
          "Test RF+diChIPMunk HUMAN  ":qualitative_colors[4], 
          "Test RF+SLIM m=1 HUMAN  ":qualitative_colors[4],
          "Test RF+SLIM m=-5 HUMAN  ":qualitative_colors[4],
          "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ":qualitative_colors[4],
          "Test full RandomForest HUMAN  ":qualitative_colors[4], 
          #"Test full RandomForest MOUSE  ":qualitative_colors[5], 
          #"Test RandomForest MOUSE  ":qualitative_colors[5], 
          #"Test RF+diChIPMunk MOUSE  ":qualitative_colors[5], 
          #"Test RF+SLIM m=1 MOUSE  ":qualitative_colors[5],
          #"Test RF+SLIM m=-5 MOUSE  ":qualitative_colors[5],
          #"Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  ":qualitative_colors[5]
}

ax = sns.swarmplot(x="value", y="variable",
                   data=df_melted,
                   size=7, zorder=2, palette="tab10")

ax = sns.boxplot(x="value", y="variable", 
                 data=df_melted,
                 fill=False, linewidth=3, zorder=3, color="black", showcaps=False, fliersize=0)


results = df_melted.groupby('variable').apply(lambda group: wilcoxon(group['value'], alternative='greater')).reset_index()
results.columns = ['variable', 'wilcoxon_result']
results['statistic'], results['p_value'] = zip(*results['wilcoxon_result'])
df = df_melted
i=0
for variable in [x[0] for x in list(my_pal.items())]:
    p_value = float(results[results['variable'] == variable]['p_value'])
    y = df[df['variable'] == variable]['value'].max() + 0.02 
    print(variable, p_value)
    if p_value < 0.001:
        plt.text(y, i, '***', ha='center', color='red')
    elif p_value < 0.01:
        plt.text(y, i, '**', ha='center', color='red')
    elif p_value < 0.05:
        plt.text(y, i, '*', ha='center', color='red')
    i+=1


ax.set_ylabel('Model')
ax.set_xlabel('Δ  AUC PRC')
ax.axvline(0, ls='-.', linewidth=3, zorder=1, color="black")
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

plt.savefig(f"{os.path.join(cfg['paths']['output_dir'], '..', 'Overall_improvement_HUMAN_PR_all.pdf')}", dpi='figure', pad_inches=1, bbox_inches='tight', transparent=True)


In [ ]:
#SLIM_df = pd.read_csv(outputdir + "/" + "HUMAN_MOUSE_SLIM.txt", header=None, sep='\t')

# SLIM_df = pd.read_csv(os.path.join(cfg['paths']['output_dir'], 'outputdir', 'HUMAN_MOUSE_SLIM_roc_mono_di_RandomForestClassifier.txt'), header=None, sep='\t')
# #SLIM_df.columns = ["TF_name", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ", "Train diChIPMunk HUMAN  ", "Test SLIM m=1 MOUSE  ", "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM 1 MOUSE  ", "Test RF+SLIM -5 MOUSE  ", "Test RF+SLIM 1,-5 +diChIPMunk MOUSE  "]
# SLIM_df.columns = ["TF_name", "Best PWM #", "Best PWM name", "Train PWM HUMAN  ", "Test PWM HUMAN  ", "Test PWM MOUSE  ", "Train SLIM m=0 HUMAN  ", "Train SLIM m=1 HUMAN  ", "Train SLIM m=-5 HUMAN  ","Train diChIPMunk HUMAN  ", "Test SLIM m=0 HUMAN  ", "Test SLIM m=1 HUMAN  ",  "Test SLIM m=-5 HUMAN  ", "Test diChIPMunk HUMAN  ", "Test SLIM m=0 MOUSE  ", "Test SLIM m=1 MOUSE  ",  "Test SLIM m=-5 MOUSE  ", "Test diChIPMunk MOUSE  ", "Train full RandomForest HUMAN  ", "Train RandomForest HUMAN  ", "Train RF+diChIPMunk HUMAN  ", "Train RF+SLIM m=1 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest HUMAN  ", "Test RandomForest HUMAN  ", "Test RF+diChIPMunk HUMAN  ", "Test RF+SLIM m=1 HUMAN  ", "Test RF+SLIM m=-5 HUMAN  ", "Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ", "Test full RandomForest MOUSE  ", "Test RandomForest MOUSE  ", "Test RF+diChIPMunk MOUSE  ", "Test RF+SLIM m=1 MOUSE  ", "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "]

SLIM_df1 = pd.DataFrame()
#SLIM_df1["Train SLIM m=0 HUMAN  "] = SLIM_df["Train SLIM m=0 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=1 HUMAN  "] = SLIM_df["Train SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-5 HUMAN  "] = SLIM_df["Train SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train SLIM m=-7 HUMAN  "] = SLIM_df["Train SLIM m=-7 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train diChIPMunk HUMAN  "] = SLIM_df["Train diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Test SLIM m=0 HUMAN  "] = SLIM_df["Test SLIM m=0 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=1 HUMAN  "] = SLIM_df["Test SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=-5 HUMAN  "] = SLIM_df["Test SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test SLIM m=-7 HUMAN  "] = SLIM_df["Test SLIM m=-7 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test diChIPMunk HUMAN  "] = SLIM_df["Test diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test SLIM m=0 MOUSE  "] = SLIM_df["Test SLIM m=0 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test SLIM m=1 MOUSE  "] = SLIM_df["Test SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test SLIM m=-5 MOUSE  "] = SLIM_df["Test SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Test SLIM m=-7 MOUSE  "] = SLIM_df["Test SLIM m=-7 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test diChIPMunk MOUSE  "] = SLIM_df["Test diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df1["Train full RandomForest HUMAN  "] = SLIM_df["Train full RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RandomForest HUMAN  "] = SLIM_df["Train RandomForest HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+diChIPMunk HUMAN  "] = SLIM_df["Train RF+diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1 HUMAN  "] = SLIM_df["Train RF+SLIM m=1 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=-5 HUMAN  "] = SLIM_df["Train RF+SLIM m=-5 HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Train PWM HUMAN  "]
#SLIM_df1["Test full RandomForest HUMAN  "] = SLIM_df["Test full RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RandomForest HUMAN  "] = SLIM_df["Test RandomForest HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RF+diChIPMunk HUMAN  "] = SLIM_df["Test RF+diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RF+SLIM m=1 HUMAN  "] = SLIM_df["Test RF+SLIM m=1 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RF+SLIM m=-5 HUMAN  "] = SLIM_df["Test RF+SLIM m=-5 HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
#SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  "] - SLIM_df["Test PWM HUMAN  "]
SLIM_df1["Test RandomForest MOUSE  "] = SLIM_df["Test RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test RF+diChIPMunk MOUSE  "] = SLIM_df["Test RF+diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test RF+SLIM m=1 MOUSE  "] = SLIM_df["Test RF+SLIM m=1 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test RF+SLIM m=-5 MOUSE  "] = SLIM_df["Test RF+SLIM m=-5 MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] = SLIM_df["Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
SLIM_df1["Test full RandomForest MOUSE  "] = SLIM_df["Test full RandomForest MOUSE  "] - SLIM_df["Test PWM MOUSE  "]
#SLIM_df=SLIM_df.drop(["Train PWM HUMAN  ", "Test PWM MOUSE  ", "Test PWM HUMAN  ", 
#                      "Test RF+SLIM m=-5 MOUSE  ", "Test RF+SLIM m=-5 HUMAN  ", "Train RF+SLIM m=-5 HUMAN  "], axis=1)

SLIM_df1["TF_name"] = SLIM_df["TF_name"]
df_melted = pd.melt(SLIM_df1, id_vars=["TF_name"])

import seaborn as sns


qualitative_colors = sns.color_palette("Set3", 7)
qualitative_colors
qualitative_colors_index = [0,0,0,1,1,1,2,2,2,3,3,3,3,3,4,4,4,4,4,5,5,5,5,5]
print(len(qualitative_colors_index))

sns.set(font_scale=4, style="ticks", font="Lato")
sns.set_style({"xtick.direction": "in","ytick.direction": "in"})
matplotlib.rcParams['font.weight'] = "medium"
matplotlib.rcParams['axes.labelweight'] = 'medium'
matplotlib.rcParams['figure.titleweight'] = 'medium'
matplotlib.rcParams['axes.titleweight'] = 'medium'
matplotlib.rcParams['figure.figsize'] = 10, 20
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42


plt.title("", fontsize=30)

my_pal = {#"Train SLIM m=0 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=1 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=-5 HUMAN  ":qualitative_colors[0], 
          #"Train SLIM m=-7 HUMAN  ":qualitative_colors[0], 
          #"Train diChIPMunk HUMAN  ":qualitative_colors[0],
          #"Test SLIM m=0 HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=1 HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=-5 HUMAN  ":qualitative_colors[1], 
          #"Test SLIM m=-7 HUMAN  ":qualitative_colors[1], 
          #"Test diChIPMunk HUMAN  ":qualitative_colors[1], 
          "Test SLIM m=0 MOUSE  ":qualitative_colors[2], 
          "Test SLIM m=1 MOUSE  ":qualitative_colors[2], 
          "Test SLIM m=-5 MOUSE  ":qualitative_colors[2], 
          #"Test SLIM m=-7 MOUSE  ":qualitative_colors[2], 
          "Test diChIPMunk MOUSE  ":qualitative_colors[2],
          #"Train full RandomForest HUMAN  ":qualitative_colors[3], 
          #"Train RandomForest HUMAN  ":qualitative_colors[3], 
          #"Train RF+diChIPMunk HUMAN  ":qualitative_colors[3], 
          #"Train RF+SLIM m=1 HUMAN  ":qualitative_colors[3],
          #"Train RF+SLIM m=-5 HUMAN  ":qualitative_colors[3],
          #"Train RF+SLIM m=1,-5 +diChIPMunk HUMAN  ":qualitative_colors[3],
          #"Test full RandomForest HUMAN  ":qualitative_colors[4], 
          #"Test RandomForest HUMAN  ":qualitative_colors[4], 
          #"Test RF+diChIPMunk HUMAN  ":qualitative_colors[4], 
          #"Test RF+SLIM m=1 HUMAN  ":qualitative_colors[4],
          #"Test RF+SLIM m=-5 HUMAN  ":qualitative_colors[4],
          #"Test RF+SLIM m=1,-5 +diChIPMunk HUMAN  ":qualitative_colors[4],
          "Test RandomForest MOUSE  ":qualitative_colors[5], 
          "Test RF+diChIPMunk MOUSE  ":qualitative_colors[5], 
          "Test RF+SLIM m=1 MOUSE  ":qualitative_colors[5],
          "Test RF+SLIM m=-5 MOUSE  ":qualitative_colors[5],
          "Test RF+SLIM m=1,-5 +diChIPMunk MOUSE  ":qualitative_colors[5],
          "Test full RandomForest MOUSE  ":qualitative_colors[5]
          }


ax = sns.swarmplot(x="value", y="variable",
                   data=df_melted,
                   size=7, zorder=2, palette="tab10")

ax = sns.boxplot(x="value", y="variable", 
                 data=df_melted,
                 fill=False, linewidth=3, zorder=3, color="black", showcaps=False, fliersize=0)


results = df_melted.groupby('variable').apply(lambda group: wilcoxon(group['value'], alternative='greater')).reset_index()
results.columns = ['variable', 'wilcoxon_result']
results['statistic'], results['p_value'] = zip(*results['wilcoxon_result'])

df = df_melted
i=0
for variable in [x[0] for x in list(my_pal.items())]:
    p_value = float(results[results['variable'] == variable]['p_value'])
    y = df[df['variable'] == variable]['value'].max() + 0.02 
    print(variable, p_value)
    if p_value < 0.001:
        plt.text(y, i, '***', ha='center', color='red')
    elif p_value < 0.01:
        plt.text(y, i, '**', ha='center', color='red')
    elif p_value < 0.05:
        plt.text(y, i, '*', ha='center', color='red')
    i+=1



ax.set_ylabel('Model')
ax.set_xlabel('Δ  AUC PRC')
ax.axvline(0, ls='-.', linewidth=3, zorder=1, color="black")

plt.savefig(f"{os.path.join(cfg['paths']['output_dir'], '..', 'Overall_improvement_MOUSE_PR_all.pdf')}", dpi='figure', pad_inches=1, bbox_inches='tight', transparent=True)

In [ ]:
row